# 02 — TiNA Preprocessing and EDA

**Goal:** create a documented, leakage-safe cleaning workflow and explore the bounded time-series period selected in notebook 01.

> Replace the placeholder time column and period with the verified choices from notebook 01. Do not interpolate across maintenance shutdowns without a defensible rule.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data_utils import resolve_data_path, scan_tina, validate_event_columns

DATA_PATH = resolve_data_path()
MAX_ROWS = int(os.environ.get('TINA_EDA_ROWS', '1000000'))
RANDOM_SEED = 613
TIME_COL = os.environ.get('TINA_TIME_COLUMN', 'unix')  # verified source field

## Load the documented analysis period

For the capstone run, replace `head` with an explicit timestamp filter so the same period is selected even if the source file changes order.

In [ ]:
lazy = scan_tina(DATA_PATH)
validate_event_columns(lazy)
frame = lazy.head(MAX_ROWS).collect()
print(f'Loaded shape: {frame.shape}')
if TIME_COL not in frame.columns:
    raise KeyError(f'Set TINA_TIME_COLUMN to the verified timestamp column. Available: {frame.columns}')
frame = frame.sort(TIME_COL)

## Structural checks

In [ ]:
duplicate_rows = frame.is_duplicated().sum()
protected_columns = {TIME_COL, 'm_id', 'm_subid', 'alarms'}
constant_columns = [
    column for column in frame.columns
    if column not in protected_columns and frame[column].n_unique() <= 1
]
null_summary = pl.DataFrame({
    'column': frame.columns,
    'null_pct': [frame[column].null_count() / frame.height * 100 for column in frame.columns],
}).sort('null_pct', descending=True)
print(f'Exact duplicate rows: {duplicate_rows:,}')
print(f'Constant columns: {constant_columns}')
null_summary.head(20)

## Cleaning policy

The starter policy removes exact duplicate rows and preview-constant fields. Missing sensor values are left missing here: imputation must be fitted on training data only inside the modelling pipeline. Do not replace maintenance or alarm labels with imputed values.

In [ ]:
clean = frame.unique(maintain_order=True).drop(constant_columns)
print(f'Cleaned shape: {clean.shape}')

## Event prevalence and time-series inspection

In [ ]:
event_rates = clean.select(
    (pl.col('m_id').cast(pl.String).str.to_lowercase() != 'none').mean().alias('maintenance_row_rate'),
    (pl.col('alarms').cast(pl.String).str.to_lowercase() != 'none').mean().alias('alarm_row_rate'),
)
event_rates

In [ ]:
numeric_columns = [
    name for name, dtype in clean.schema.items()
    if dtype.is_numeric() and name not in {TIME_COL, 'm_id', 'm_subid', 'alarms'}
]
plot_columns = numeric_columns[:6]
plot_frame = clean.select([TIME_COL, *plot_columns]).head(20000).to_pandas()
plot_frame.set_index(TIME_COL)[plot_columns].plot(subplots=True, figsize=(14, 10), legend=False)
plt.suptitle('Selected TiNA sensor signals (bounded view)')
plt.tight_layout()

## Chronological split boundaries

In [ ]:
n_rows = clean.height
train_end = int(n_rows * 0.70)
validation_end = int(n_rows * 0.85)
split_summary = pl.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [train_end, validation_end - train_end, n_rows - validation_end],
    'start_time': [clean[TIME_COL][0], clean[TIME_COL][train_end], clean[TIME_COL][validation_end]],
    'end_time': [clean[TIME_COL][train_end - 1], clean[TIME_COL][validation_end - 1], clean[TIME_COL][-1]],
})
split_summary

## EDA conclusions

- Duplicate/constant/missing-field decision: **TODO**
- Sensor discontinuities and shutdown patterns: **TODO**
- Maintenance and alarm imbalance: **TODO**
- Proposed resampling frequency and justification: **TODO**
- Final chronological split dates and event counts: **TODO**
- Risks to revisit during feature engineering: **TODO**